# Lecture 16 · Positional Encoding 유무 비교

같은 서브워드 Self-Attention 모델에서 **Positional Encoding(PE)의 유무만 변경**하고 학습 loss를 비교합니다.

- 두 모델은 같은 지역 어휘와 같은 학습 데이터를 사용합니다.
- 두 모델은 동일한 초기 가중치에서 시작합니다.
- causal mask와 padding mask를 모두 적용합니다.
- `[PAD]` 정답은 loss 계산에서 제외합니다.

이 실습의 목적은 PE가 항상 더 낮은 loss를 보장한다고 결론 내리는 것이 아니라, **위치 정보를 추가했을 때 학습 양상이 어떻게 달라지는지 관찰하는 것**입니다.

## 1. 라이브러리와 설정

In [ ]:
!pip install -q transformers koreanize-matplotlib

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import koreanize_matplotlib

from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoTokenizer

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

D_MODEL = 32
N_HEADS = 2
EPOCHS = 50
BATCH_SIZE = 8

assert D_MODEL % N_HEADS == 0


## 2. 고유 문장 30개 준비

문장을 반복 복제하지 않고 Lecture 15와 같은 30개 고유 문장을 사용합니다.

In [ ]:
sentences = [
    "고양이가 매트 위에 앉았다 그것이 부드러워서",
    "고양이가 소파에 누웠다 그것이 따뜻해서",
    "강아지가 공원으로 달려갔다 그것이 넓어서",
    "강아지가 마당에서 뛰어놀았다 그것이 즐거워서",
    "새가 나무로 날아갔다 그것이 높아서",
    "새가 하늘을 날았다 그것이 시원해서",
    "토끼가 풀밭에서 뛰었다 그것이 푹신해서",
    "토끼가 굴로 들어갔다 그것이 아늑해서",
    "다람쥐가 나무를 올랐다 그것이 튼튼해서",
    "다람쥐가 도토리를 모았다 그것이 맛있어서",
    "곰이 강으로 걸어갔다 그것이 맑아서",
    "곰이 동굴에서 잠들었다 그것이 따뜻해서",
    "여우가 숲으로 들어갔다 그것이 조용해서",
    "여우가 언덕을 넘었다 그것이 가팔라서",
    "사슴이 초원에서 뛰었다 그것이 드넓어서",
    "사슴이 시냇물을 건넜다 그것이 시원해서",
    "펭귄이 얼음판에서 미끄러졌다 그것이 매끄러워서",
    "펭귄이 바다로 뛰어들었다 그것이 차가워서",
    "코끼리가 호수로 들어갔다 그것이 깊어서",
    "코끼리가 진흙을 밟았다 그것이 부드러워서",
    "원숭이가 나뭇가지를 잡았다 그것이 단단해서",
    "원숭이가 바나나를 먹었다 그것이 달콤해서",
    "판다가 대나무를 씹었다 그것이 신선해서",
    "판다가 언덕에서 굴렀다 그것이 재미있어서",
    "고래가 바다를 헤엄쳤다 그것이 광활해서",
    "고래가 물살을 갈랐다 그것이 세차서",
    "부엉이가 나뭇가지에 앉았다 그것이 튼튼해서",
    "부엉이가 밤하늘을 날았다 그것이 고요해서",
    "거북이가 모래밭을 기어갔다 그것이 따뜻해서",
    "거북이가 바위 위에 올랐다 그것이 널찍해서",
]

print("고유 문장 수:", len(sentences))
print("첫 번째 문장:", sentences[0])


## 3. KLUE 토크나이저로 서브워드 분리

KLUE-BERT 모델을 학습하는 것이 아니라, KLUE 토크나이저만 서브워드 분리에 사용합니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

token_lists = []
for sentence in sentences:
    tokens = tokenizer.tokenize(sentence)
    tokens.append("[EOS]")
    token_lists.append(tokens)

print("원문:", sentences[0])
print("서브워드:", token_lists[0])


## 4. 실습용 지역 어휘 만들기

전체 KLUE 어휘 대신 30개 문장에 실제로 등장한 서브워드만 사용합니다.

In [ ]:
all_tokens = sorted({
    token
    for tokens in token_lists
    for token in tokens
})

token_to_id = {"[PAD]": 0, "[UNK]": 1}
for token in all_tokens:
    if token not in token_to_id:
        token_to_id[token] = len(token_to_id)

id_to_token = {
    token_id: token
    for token, token_id in token_to_id.items()
}

pad_id = token_to_id["[PAD]"]
vocab_size = len(token_to_id)

print("실습 어휘 크기:", vocab_size)
print("어휘 예시:", list(token_to_id.items())[:20])


## 5. 입력과 한 칸 이동한 정답 만들기

+`sample_weight`가 0인 `[PAD]` 위치는 loss 계산에서 제외됩니다.

In [ ]:
max_tokens = max(len(tokens) for tokens in token_lists)
SEQ_LEN = max_tokens - 1

X = []
y = []

for tokens in token_lists:
    ids = [token_to_id[token] for token in tokens]
    ids = ids + [pad_id] * (max_tokens - len(ids))
    X.append(ids[:-1])
    y.append(ids[1:])

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)
sample_weight = (y != pad_id).astype("float32")

print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("유효한 정답 수:", int(sample_weight.sum()))


In [ ]:
valid_n = int(sample_weight[0].sum())

input_tokens = [id_to_token[token_id] for token_id in X[0, :valid_n]]
target_tokens = [id_to_token[token_id] for token_id in y[0, :valid_n]]

print("입력 X:", input_tokens)
print("정답 y:", target_tokens)


## 6. Sin/Cos Positional Encoding 만들기

In [ ]:
def build_sinusoidal_pe(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]

    angles = positions / np.power(
        10000,
        (2 * (dimensions // 2)) / d_model
    )

    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])

    return angles.astype("float32")

pe_matrix = build_sinusoidal_pe(SEQ_LEN, D_MODEL)
print("PE 크기:", pe_matrix.shape)


## 7. PE 유무만 다른 두 모델 만들기

+모델 구조와 초기 가중치는 같고, 토큰 임베딩에 PE를 더하는지 여부만 다릅니다.

In [ ]:
def build_model(use_pe, model_name):
    token_input = keras.Input(
        shape=(SEQ_LEN,),
        dtype="int32",
        name="tokens"
    )

    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=D_MODEL,
        name="token_embedding"
    )(token_input)

    if use_pe:
        x = keras.ops.add(x, pe_matrix)

    pad_mask = keras.ops.not_equal(token_input, pad_id)
    pad_mask = keras.ops.expand_dims(pad_mask, axis=1)

    attention_output = layers.MultiHeadAttention(
        num_heads=N_HEADS,
        key_dim=D_MODEL // N_HEADS,
        name="self_attention"
    )(
        x,
        x,
        attention_mask=pad_mask,
        use_causal_mask=True
    )

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization(name="normalization")(x)
    output = layers.Dense(vocab_size, name="lm_head")(x)

    model = keras.Model(token_input, output, name=model_name)
    model.compile(
        optimizer="adam",
        loss=keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        )
    )
    return model

model_no_pe = build_model(False, "without_pe")
model_with_pe = build_model(True, "with_pe")

# 공정한 비교: 두 모델을 동일한 초기 가중치로 맞춥니다.
model_with_pe.set_weights(model_no_pe.get_weights())

print("초기 가중치 동일:", all(
    np.array_equal(a, b)
    for a, b in zip(model_no_pe.get_weights(), model_with_pe.get_weights())
))


## 8. 같은 조건으로 두 모델 학습하기

In [ ]:
print("[PE 없음]")
history_no_pe = model_no_pe.fit(
    X,
    y,
    sample_weight=sample_weight,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    verbose=2
)

print("\n[PE 있음]")
history_with_pe = model_with_pe.fit(
    X,
    y,
    sample_weight=sample_weight,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    verbose=2
)


## 9. PE 패턴과 loss 비교

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
image = ax.imshow(
    pe_matrix.T,
    aspect="auto",
    cmap="RdBu",
    origin="lower"
)
ax.set_xlabel("위치")
ax.set_ylabel("차원")
ax.set_title("Positional Encoding (sin/cos)")
plt.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history_no_pe.history["loss"], label="PE 없음")
ax.plot(history_with_pe.history["loss"], label="PE 있음")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("동일 초기 가중치에서 PE 유무 비교")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("PE 없음 최종 loss:", round(history_no_pe.history["loss"][-1], 4))
print("PE 있음 최종 loss:", round(history_with_pe.history["loss"][-1], 4))


## 10. 결과 해석

- 두 모델은 동일한 데이터, 초기 가중치, 배치 순서에서 학습했습니다.
- 따라서 주요 실험 차이는 Positional Encoding의 사용 여부입니다.
- 다만 30개 문장만 사용한 toy 실험이므로 loss 차이를 일반적인 성능 우위로 확대 해석하지 않습니다.